# Exploratory Data Analysis — Transaction Behaviour

Uses the analytics workspace established in PR-007 to ask questions of the
transaction data and document what it actually shows, before any
feature engineering or modelling is built on top of it.

**Important principle.** The UI prototype for this product surfaces concepts
like spending trends, unusual purchases, recurring subscriptions, category
changes and savings rate. Those are product concepts the prototype was
designed around using generated, deterministic data — not evidence from this
dataset. Here they are treated as *questions to investigate*, and the answer
"there isn't enough data to say" is a valid, honest finding in its own
right, not a failure of the analysis.

Dataset: `data/raw/finance_analytics_test_transactions.csv` — the same
synthetic fixture used in notebook 01, with 22 rows, a duplicate
transaction and three deliberately broken QA rows.

In [1]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from finance_analytics.analysis.category import category_summary
from finance_analytics.analysis.merchant import merchant_summary
from finance_analytics.analysis.outliers import (
    flag_category_relative_outliers,
    flag_iqr_outliers,
    robust_zscores,
)
from finance_analytics.analysis.recurring import recurring_candidates
from finance_analytics.analysis.temporal import add_temporal_features
from finance_analytics.data.quality import build_quality_report
from finance_analytics.duckdb_queries import connect_with_transactions
from finance_analytics.io.csv import load_transactions_csv
from finance_analytics.validation.transactions import validate_transactions

pd.set_option("display.max_columns", None)

DATA_PATH = Path("../data/raw/finance_analytics_test_transactions.csv")

## 1. Research Questions

| # | Question |
|---|---|
| RQ1 | How does spending evolve over time? |
| RQ2 | Which categories drive spending? |
| RQ3 | What does transaction behaviour look like (distribution, not just averages)? |
| RQ4 | Which merchants dominate spending? |
| RQ5 | Are there recurring transactions? |
| RQ6 | Are there unusual transactions? |
| RQ7 | What can we learn about income and savings? |

Each is investigated below with evidence from this dataset. None of them
are answered with a machine-learning model, a production anomaly detector
or financial advice — see PR-008's Out of Scope.

## 2. Dataset Overview

Load the raw CSV exactly as notebook 01 did, and get a first look at its
shape and columns.

In [2]:
transactions = load_transactions_csv(DATA_PATH)

print(f"Rows: {len(transactions)} | Columns: {list(transactions.columns)}")
transactions

Rows: 22 | Columns: ['id', 'date', 'amount', 'currency', 'description', 'merchant', 'category', 'account']


,id,date,amount,currency,description,merchant,category,account
0,1,2026-02-02,-12.50,EUR,Morning coffee,Coffee Corner,Food & Dining,Main Account
1,2,2026-02-03,-54.90,EUR,Weekly groceries,Continente,Groceries,Main Account
2,3,2026-02-05,-29.99,EUR,Monthly subscription,Spotify,Subscriptions,Main Account
3,4,2026-02-07,-82.40,EUR,Dinner with friends,O Pescador,Food & Dining,Main Account
4,5,2026-02-10,-45.00,EUR,Electricity bill,EDP,Utilities,Main Account
5,6,2026-02-12,-18.75,EUR,Pharmacy,Farmácia Central,Health,Main Account
6,7,2026-02-15,-120.00,EUR,Train tickets,CP,Transport,Main Account
7,8,2026-02-18,-642.50,EUR,"Flight, Lisbon to Rome",TAP Air,Travel,Main Account
8,9,2026-02-20,-899.00,EUR,New laptop purchase,MediaMarkt,Shopping,Main Account
9,10,2026-02-22,-8.20,EUR,"Lunch, ""daily menu""",Café Central,Food & Dining,Main Account


## 3. Data Quality

Notebook 01 already validated this dataset in detail (missing columns,
invalid dates/amounts, missing values) and produced a full data-quality
report. This section only re-derives what's needed to build a **clean**
working set for behavioural analysis — it is not a replacement for
notebook 01.

In [3]:
validation = validate_transactions(transactions)
quality_report = build_quality_report(transactions)

print(f"Valid: {validation.is_valid}")
quality_report.to_frame()

Valid: False


,value
Rows,22
Columns,8
Duplicate rows,1
Duplicate transaction IDs,1
Invalid dates,1
Invalid amounts,1
Unique merchants,16
Unique categories,10
Date range,2026-02-02 to 2026-03-19
Income transactions,2


Building `clean` from `transactions` takes three explicit steps:

1. **Drop rows with an invalid date or amount** (row 19: `not-a-date`; row
   20: `not-an-amount`) — behavioural analysis needs both to be usable.
2. **Drop duplicate transaction IDs** (row 18 duplicates row 14, a repeated
   Spotify charge) — counting it twice would inflate spend and transaction
   volume.
3. **Drop rows that are themselves QA fixtures, not transactions.** Rows
   19–21 all have `test` in their `description` — they exist to give the
   PR-007 validator something to catch, not to describe real spending. Row
   19 and 20 are already gone after step 1; row 21 ("Missing merchant
   test") has a *structurally valid* date and amount, so it survives that
   filter and needs an explicit, documented exclusion rather than an
   accidental one. Left in, it would sit alone in the `Shopping` category
   and quietly distort that category's numbers in section 6.

In [4]:
structurally_valid = transactions.dropna(subset=["date", "amount"]).drop_duplicates(subset=["id"])

synthetic_qa_mask = structurally_valid["description"].str.contains("test", case=False, na=False)
dropped_qa_rows = structurally_valid.loc[synthetic_qa_mask]

clean = structurally_valid.loc[~synthetic_qa_mask].copy()

print(f"Raw rows:                    {len(transactions)}")
print(f"Structurally valid, deduped: {len(structurally_valid)}")
print(f"Dropped as QA fixtures:      {list(dropped_qa_rows['id'])}")
print(f"Clean rows for this notebook: {len(clean)}")
dropped_qa_rows

Raw rows:                    22
Structurally valid, deduped: 19
Dropped as QA fixtures:      ['21']
Clean rows for this notebook: 18


,id,date,amount,currency,description,merchant,category,account
21,21,2026-03-19,-10.0,EUR,Missing merchant test,NaN,Shopping,Main Account


`clean` (18 rows) is the dataset used for the rest of this notebook.

## 4. Temporal Behaviour — RQ1

Add reusable calendar features (`year`, `month`, `month_period`,
`day_of_week`, `day_of_month`) via
`finance_analytics.analysis.temporal.add_temporal_features`, then register
`clean` with DuckDB for the rest of the notebook.

In [5]:
clean = add_temporal_features(clean)
con = connect_with_transactions(clean)

clean[["date", "month_period", "day_of_week", "day_of_month"]].head()

,date,month_period,day_of_week,day_of_month
0,2026-02-02,2026-02,Monday,2
1,2026-02-03,2026-02,Tuesday,3
2,2026-02-05,2026-02,Thursday,5
3,2026-02-07,2026-02,Saturday,7
4,2026-02-10,2026-02,Tuesday,10


**Monthly income, expenses, net cash flow and volume.** A non-trivial
DuckDB query — grouping and conditional aggregation read more clearly in
SQL here than as a multi-step pandas groupby.

In [6]:
monthly_cashflow = con.execute(
    """
    SELECT
        month_period AS month,
        SUM(CASE WHEN amount > 0 THEN amount ELSE 0 END) AS income,
        SUM(CASE WHEN amount < 0 THEN amount ELSE 0 END) AS expenses,
        SUM(amount) AS net,
        COUNT(*) AS transaction_count
    FROM transactions
    GROUP BY month_period
    ORDER BY month_period
    """
).df()

monthly_cashflow["savings_rate"] = monthly_cashflow["net"] / monthly_cashflow["income"]
monthly_cashflow["net_change"] = monthly_cashflow["net"].diff()
monthly_cashflow

,month,income,expenses,net,transaction_count,savings_rate,net_change
0,2026-02,3450.0,-1929.23,1520.77,12,0.440803,NaN
1,2026-03,500.0,-251.49,248.51,6,0.497020,-1272.26


**Observed:** February (12 transactions, a full month in the data) has net
+€1,520.77 on income of €3,450. March (6 transactions) has net +€248.51 on
income of €500 — but the data ends on 2026-03-19, so March is a **partial
month, not a shorter one**. Its `net_change` of -€1,272.26 versus February
is not a real month-over-month decline; it's an artifact of comparing a
full month to roughly two-thirds of one. Month-over-month comparison is not
meaningful again until a second full month of data exists.

**Weekday behaviour.** With 18 transactions spread over 7 weekdays, there
isn't enough volume to say anything reliable about weekday-vs-weekend
spending — a category with 2–3 observations per weekday is noise, not a
pattern.

In [7]:
clean["day_of_week"].value_counts()

day_of_week
Tuesday      5
Thursday     4
Sunday       4
Saturday     2
Monday       1
Wednesday    1
Friday       1
Name: count, dtype: int64

Visualisation: monthly income vs expenses.

In [8]:
fig = go.Figure()
fig.add_bar(name="Income", x=monthly_cashflow["month"], y=monthly_cashflow["income"])
fig.add_bar(name="Expenses", x=monthly_cashflow["month"], y=monthly_cashflow["expenses"].abs())
fig.update_layout(
    barmode="group",
    title="Monthly Income vs Expenses",
    yaxis_title="EUR",
    xaxis_title="",
)
fig.show()

## 5. Spending Distribution — RQ3

Expense amounts only (`Income` is a different kind of thing and would
distort a spending distribution). Mean vs median is the first check for
skew — don't rely on the mean alone.

In [9]:
expenses = clean.loc[clean["amount"] < 0].copy()
expenses["abs_amount"] = expenses["amount"].abs()

expenses["abs_amount"].describe()

count     16.000000
mean     136.295000
std      254.077758
min        8.200000
25%       21.487500
50%       41.400000
75%       86.800000
max      899.000000
Name: abs_amount, dtype: float64

**Observed:** mean (€136.30) is more than 3x the median (€41.40) — a
strongly right-skewed distribution. That's driven by two one-off purchases
(MediaMarkt €899.00, TAP Air €642.50); without them the mean would sit much
closer to the median. Whenever a dataset has this shape, the mean alone is
misleading — it describes a typical transaction that barely occurs.

In [10]:
bins = pd.cut(
    expenses["abs_amount"],
    bins=[0, 25, 100, float("inf")],
    labels=["< €25", "€25 – €100", "≥ €100"],
    right=False,
)
size_split = expenses.groupby(bins, observed=True).agg(
    transaction_count=("abs_amount", "count"), total_spend=("abs_amount", "sum")
)
size_split["share_of_spend"] = size_split["total_spend"] / expenses["abs_amount"].sum()
size_split

,transaction_count,total_spend,share_of_spend
abs_amount,,,
< €25,5,77.84,0.035695
€25 – €100,7,341.38,0.156545
≥ €100,4,1761.50,0.807761


**Observed:** transactions under €25 are 31% of expense transactions but
only ~3.6% of expense value. Transactions of €100+ are 25% of transactions
but ~80.8% of value. Most of what leaves the account in this window is a
small number of large, infrequent transactions — not the accumulation of
many small ones.

**Distribution by category** is deliberately shown as a table rather than a
box plot: most categories have 1–4 transactions here, and a box plot on
that few points would look precise while being nearly meaningless.

In [11]:
expenses.groupby("category")["abs_amount"].agg(
    transaction_count="count", min="min", median="median", max="max"
).sort_values("transaction_count", ascending=False)

,transaction_count,min,median,max
category,,,,
Food & Dining,4,8.20,25.15,82.40
Subscriptions,3,15.99,29.99,29.99
Groceries,2,54.90,58.10,61.30
Transport,2,22.40,71.20,120.00
Cash,1,100.00,100.00,100.00
Health,1,18.75,18.75,18.75
Shopping,1,899.00,899.00,899.00
Travel,1,642.50,642.50,642.50
Utilities,1,45.00,45.00,45.00


Visualisation: distribution of transaction amounts.

In [12]:
fig = px.histogram(
    expenses,
    x="abs_amount",
    nbins=10,
    marginal="rug",
    title="Distribution of Transaction Amounts (Expenses)",
    labels={"abs_amount": "Transaction amount (EUR)"},
)
fig.update_layout(showlegend=False)
fig.show()

## 6. Category Behaviour — RQ2

`finance_analytics.analysis.category.category_summary` builds the
category-level table (expenses only — see section 3's reasoning for why
`Income` doesn't belong in a "what drives spending" view).

In [13]:
categories = category_summary(clean)
categories

,category,transaction_count,total_spend,mean_transaction,median_transaction,share_of_spend,monthly_volatility
0,Shopping,1,899.00,899.000000,899.00,0.412249,NaN
1,Travel,1,642.50,642.500000,642.50,0.294627,NaN
2,Transport,2,142.40,71.200000,71.20,0.065300,0.969292
3,Food & Dining,4,140.90,35.225000,25.15,0.064612,0.655416
4,Groceries,2,116.20,58.100000,58.10,0.053285,0.077891
5,Cash,1,100.00,100.000000,100.00,0.045856,NaN
6,Subscriptions,3,75.97,25.323333,29.99,0.034837,0.297661
7,Utilities,1,45.00,45.000000,45.00,0.020635,NaN
8,Health,1,18.75,18.750000,18.75,0.008598,NaN


**Observed:** `Shopping` (41.2% of spend) and `Travel` (29.5%) together are
70.7% of all spend in this window — but each is a *single* transaction
(the laptop, the flight), not a repeated pattern. Calling them "categories
that drive spending" is technically true for this window and misleading
about behaviour: they say more about two one-off purchases than about how
this account is habitually used.

**Classification of monthly volatility** (coefficient of variation of
monthly totals) is only possible for categories observed in both months —
`Shopping`, `Travel`, `Cash`, `Utilities` and `Health` each appear in one
month only, so "increasing/decreasing/variable" isn't a question this
dataset can answer for them yet:

In [14]:
def classify_volatility(cv: float) -> str:
    if pd.isna(cv):
        return "insufficient data (single month)"
    if cv < 0.15:
        return "consistently high / stable"
    if cv < 0.5:
        return "moderate variation"
    return "highly variable"


categories.assign(pattern=categories["monthly_volatility"].map(classify_volatility))[
    ["category", "total_spend", "share_of_spend", "monthly_volatility", "pattern"]
]

,category,total_spend,share_of_spend,monthly_volatility,pattern
0,Shopping,899.00,0.412249,NaN,insufficient data (single month)
1,Travel,642.50,0.294627,NaN,insufficient data (single month)
2,Transport,142.40,0.065300,0.969292,highly variable
3,Food & Dining,140.90,0.064612,0.655416,highly variable
4,Groceries,116.20,0.053285,0.077891,consistently high / stable
5,Cash,100.00,0.045856,NaN,insufficient data (single month)
6,Subscriptions,75.97,0.034837,0.297661,moderate variation
7,Utilities,45.00,0.020635,NaN,insufficient data (single month)
8,Health,18.75,0.008598,NaN,insufficient data (single month)


**Observed:** `Groceries` is the most stable category (CV 0.08 — €54.90
then €61.30, a small, plausible week-to-week variation). `Subscriptions`
is moderately stable (CV 0.30 — mostly the recurring €29.99 Spotify
charge). `Food & Dining` and `Transport` are highly variable, but each is
built from only 2–4 transactions, so "highly variable" here means "not yet
enough data to see a stable pattern," not necessarily "genuinely erratic
behaviour."

Visualisations: spending by category, and category trend over time for the
categories with more than one month of data (a 2-point line is the most
"trend" this dataset supports — shown as two connected observations, not a
fitted trend).

In [15]:
category_chart = categories.sort_values("total_spend", ascending=True)
fig = px.bar(
    category_chart,
    x="total_spend",
    y="category",
    orientation="h",
    title="Spending by Category",
    labels={"total_spend": "Total spend (EUR)", "category": ""},
)
fig.update_layout(showlegend=False)
fig.show()

In [16]:
multi_month_categories = categories.loc[categories["monthly_volatility"].notna(), "category"]
monthly_category_spend = (
    clean.loc[(clean["amount"] < 0) & (clean["category"].isin(multi_month_categories))]
    .assign(spend=lambda df: df["amount"].abs())
    .groupby(["month_period", "category"], as_index=False)["spend"]
    .sum()
)

fig = px.line(
    monthly_category_spend,
    x="month_period",
    y="spend",
    color="category",
    markers=True,
    title="Category Spend by Month (categories present in both months)",
    labels={"month_period": "", "spend": "Spend (EUR)"},
)
fig.show()

## 7. Merchant Behaviour — RQ4

`finance_analytics.analysis.merchant.merchant_summary` builds the
merchant-level table, again on expenses only.

In [17]:
merchants = merchant_summary(clean)
merchants

,merchant,transaction_count,total_spend,mean_transaction,median_transaction
0,MediaMarkt,1,899.00,899.00,899.00
1,TAP Air,1,642.50,642.50,642.50
2,O Pescador,2,120.20,60.10,60.10
3,CP,1,120.00,120.00,120.00
4,Continente,2,116.20,58.10,58.10
5,ATM,1,100.00,100.00,100.00
6,Spotify,2,59.98,29.99,29.99
7,EDP,1,45.00,45.00,45.00
8,Galp,1,22.40,22.40,22.40
9,Farmácia Central,1,18.75,18.75,18.75


**Observed:** the top two merchants by spend (`MediaMarkt`, `TAP Air`) are
each a single large transaction — few large transactions, not many small
ones. The repeat merchants (`O Pescador`, `Continente`, `Spotify`, each
seen twice) sit lower in total spend but are the ones with something to say
about *frequency* — they're the natural candidates for section 8's
recurring-transaction check, while `MediaMarkt`/`TAP Air` are not (a single
occurrence has no frequency to assess).

Visualisation: top merchants by spend.

In [18]:
top_merchants = merchants.head(8).sort_values("total_spend", ascending=True)
fig = px.bar(
    top_merchants,
    x="total_spend",
    y="merchant",
    orientation="h",
    title="Top Merchants by Spend",
    labels={"total_spend": "Total spend (EUR)", "merchant": ""},
)
fig.update_layout(showlegend=False)
fig.show()

## 8. Recurring Transactions — RQ5

`finance_analytics.analysis.recurring.recurring_candidates` builds an
*exploratory* candidate table (merchants with 2+ transactions, amount and
interval variation) — not the final recurring-payment detector; that's
future work RQ5 explicitly defers.

In [19]:
recurring_candidates(clean)

,merchant,occurrences,amount_variation,median_interval_days,interval_variation
0,Continente,2,0.077891,28.0,NaN
1,O Pescador,2,0.524741,29.0,NaN
2,Spotify,2,0.000000,28.0,NaN


**Observed:** `Spotify` is the strongest candidate — identical amount both
times (0% amount variation) and a ~28-day gap, which matches a monthly
subscription cadence. `Continente` is plausible (7.8% amount variation, a
~28-day gap) but a single repeat grocery visit a month apart is thin
evidence either way. `O Pescador`'s amount varies by ~52% — consistent
with casual dining out rather than a fixed recurring charge, despite a
similar interval.

**A structural limitation, not just a data-size one:** `interval_variation`
is `NaN` for every candidate here. With exactly 2 occurrences there is
exactly 1 interval — there's nothing to compute *variation* against. Seeing
whether a merchant's cadence is *consistent* (not just present) needs at
least 3 occurrences. That's a concrete requirement for the next iteration
of this dataset, not just "more data would help in general."

**A method limitation:** `Netflix` reads like a genuine subscription from
its description, but appears only once in this 46-day window, so
`min_occurrences=2` makes it invisible to this approach entirely. A
short observation window will always under-detect recurring transactions
relative to their true frequency.

## 9. Outlier Investigation — RQ6

Three approaches, from naive to more careful, using
`finance_analytics.analysis.outliers`.

**1. Global IQR on the signed amount** — the naive version.

In [20]:
clean.loc[flag_iqr_outliers(clean["amount"]), ["date", "merchant", "category", "amount"]]

,date,merchant,category,amount
7,2026-02-18,TAP Air,Travel,-642.5
8,2026-02-20,MediaMarkt,Shopping,-899.0
11,2026-02-28,Employer Payroll,Income,3450.0
17,2026-03-15,Freelance Client,Income,500.0


This flags `Employer Payroll`'s €3,450 salary and `Freelance Client`'s €500
payment as "outliers" alongside the two large purchases — not because
they're unusual, but because signed income and signed expenses were
compared on the same global scale, and income is naturally larger. This is
exactly the trap RQ6 warns about: **a large transaction is not
automatically an anomaly**, and a threshold that doesn't distinguish income
from expenses will misclassify normal income as one.

**2. Robust z-score on expense magnitude only** — scoped to the thing
actually being asked about (unusual *spending*, not unusual cash movement).

In [21]:
expenses["robust_z"] = robust_zscores(expenses["abs_amount"])
expenses[["date", "merchant", "category", "amount", "robust_z"]].sort_values(
    "robust_z", ascending=False
)

,date,merchant,category,amount,robust_z
8,2026-02-20,MediaMarkt,Shopping,-899.00,24.071713
7,2026-02-18,TAP Air,Travel,-642.50,16.872093
6,2026-02-15,CP,Transport,-120.00,2.206199
15,2026-03-10,ATM,Cash,-100.00,1.644826
3,2026-02-07,O Pescador,Food & Dining,-82.40,1.150817
12,2026-03-03,Continente,Groceries,-61.30,0.558567
1,2026-02-03,Continente,Groceries,-54.90,0.378927
4,2026-02-10,EDP,Utilities,-45.00,0.101047
14,2026-03-08,O Pescador,Food & Dining,-37.80,-0.101047
2,2026-02-05,Spotify,Subscriptions,-29.99,-0.320264


This is cleaner: `MediaMarkt` (z ≈ 24) and `TAP Air` (z ≈ 17) stand far
above everything else; the next tier (`CP`, `ATM`, `O Pescador`, z ≈ 1–2)
is unremarkable. Both standout transactions check out as genuine on
inspection — a laptop purchase categorised `Shopping` and a flight
categorised `Travel`, both well-formed, both plausible one-off purchases
for their category, with no sign of a data-entry error (no duplicate row,
no mismatched currency, description matches merchant and category).
Occasional large purchases in `Shopping`/`Travel` look like normal
category-specific behaviour, not anomalies.

**3. Category-relative IQR** — flag a transaction only if it's unusual
*within its own category*.

In [22]:
clean.loc[flag_category_relative_outliers(clean), ["date", "merchant", "category", "amount"]]

,date,merchant,category,amount


This returns nothing — not because there's nothing unusual, but because
most categories here have only 1–3 transactions, which makes the IQR
fences degenerate (too little data to define "usual" for that category in
the first place). An empty result from this method, on this dataset, is a
statement about sample size, not a clean bill of health.

Visualisation: transaction amounts over time, sized and coloured to make
the two standout points visible against the rest of the data.

In [23]:
fig = px.scatter(
    expenses,
    x="date",
    y="abs_amount",
    color="category",
    size="abs_amount",
    hover_data=["merchant"],
    title="Expense Amounts Over Time",
    labels={"abs_amount": "Transaction amount (EUR)", "date": ""},
)
fig.show()

## 10. Income / Expense Behaviour — RQ7

Reuses `monthly_cashflow` from section 4. This is a description of what the
data shows, not financial advice.

In [24]:
monthly_cashflow[["month", "income", "expenses", "net", "savings_rate"]]

,month,income,expenses,net,savings_rate
0,2026-02,3450.0,-1929.23,1520.77,0.440803
1,2026-03,500.0,-251.49,248.51,0.497020


**Observed:** savings rate is positive in both observed periods (44.1% in
February, 49.7% in March). As noted in section 4, March is a partial month
— its rate is not directly comparable to February's, and could still move
once a full month of March data exists.

**Income variability** can't be assessed meaningfully here: there are
exactly two income events in the entire window (one salary, one freelance
payment). €3,450 salary on 2026-02-28 and €500 freelance income on
2026-03-15 look, from their descriptions, like one recurring source and one
occasional one — but that's a hypothesis from two data points, not a
statistical finding. A useful measure of income variability needs several
months of income events to compare, which this dataset doesn't yet have.

## 11. Key Findings

### Observed

- 18 of 22 raw rows are usable behavioural data after removing a
  duplicate, an invalid date, an invalid amount and one QA fixture row
  disguised as a valid transaction (section 3).
- Expense amounts are strongly right-skewed: mean €136.30 vs median
  €41.40. Two one-off purchases (MediaMarkt €899, TAP Air €642.50) account
  for ~71% of all spend in the window (sections 5–6).
- `Groceries` and `Subscriptions` are the most stable categories
  month-to-month; `Food & Dining` and `Transport` show high variation, but
  from only 2–4 transactions each (section 6).
- `Spotify` is the strongest recurring-transaction candidate (identical
  amount, ~28-day gap); `Netflix` — plausibly also recurring — is invisible
  to a 2-occurrence-minimum method because it appears only once in this
  window (section 8).
- A naive global outlier check misclassifies salary as an anomaly; scoping
  to expense magnitude cleanly separates two genuine large purchases from
  everything else; category-relative thresholds return no signal because
  most categories have too few transactions to define "usual" (section 9).
- Savings rate is positive in both observed periods, but the second is a
  partial month and there are only two income events total (section 10).

### Interpretation

If this were a real user's data, the picture is: a small number of large,
occasional purchases dominate total spend more than habitual overspending
does; a couple of category/merchant patterns (grocery runs, a music
subscription) look genuinely recurring; and naive "biggest transactions"
outlier detection would flag the wrong things (income) unless it's scoped
to expenses and ideally to category. None of this should be read as a
conclusion about real financial behaviour — it's a read of one synthetic,
46-day fixture.

### Limitations

- **Size:** 22 raw / 18 clean transactions is too small for any of the
  statistics above to generalise.
- **Time span:** 46 days, and the second month is partial — no seasonality,
  no reliable weekday/weekend pattern, and month-over-month comparison
  isn't meaningful yet.
- **Category/merchant thinness:** most categories have 1–4 transactions and
  most merchants have exactly 1, which breaks category-relative and
  merchant-relative statistics (sections 6, 9).
- **Synthetic/test data:** this fixture exists to exercise validation and
  EDA code, not to represent real spending; three of its rows are
  deliberately broken and had to be explicitly excluded (section 3).
- **No ground truth:** there are no labelled anomalies and no labelled
  recurring transactions to check any of this analysis against.
- **No external context:** no bank category codes, no merchant metadata, no
  information about the account holder — every interpretation above is
  inferred from `description`/`merchant`/`category` text alone.
- **Single account, single currency:** nothing here generalises to
  multi-account or multi-currency behaviour.

### Next Questions

- With a full year of data, do `Groceries`/`Subscriptions` stay the stable
  categories, and does `Food & Dining`/`Transport` volatility settle down
  with more observations?
- At what per-category transaction volume does category-relative outlier
  detection start returning a real signal instead of an empty result?
- With 3+ occurrences per merchant, how consistent are the recurring
  candidates' intervals — does `Spotify` stay at exactly 0% amount
  variation, and does `Continente`/`O Pescador` clarify into "recurring" or
  "coincidental"?
- Does a lower `min_occurrences` (with a longer safety window) recover
  merchants like `Netflix` that this method currently misses?
- How does income variability look with a real spread of income events
  rather than two?

## 12. Implications for Feature Engineering

Candidate features for a future feature-engineering PR, each motivated by
something observed above. **Not implemented as a pipeline here** — this is
a list of justified candidates, not code.

| Feature | Motivation |
|---|---|
| `transaction_amount` | Base signal for every section above. |
| `log_transaction_amount` | Section 5: expense amounts are strongly right-skewed (mean 3x median); a log transform would make the distribution far more tractable for downstream statistics/models. |
| `category_share` | Section 6: `share_of_spend` already distinguishes categories that dominate spend by volume vs by one-off size — worth having per-transaction. |
| `merchant_frequency` | Section 7/8: transaction count per merchant is what separated "many small transactions" merchants from "few large" ones, and fed directly into recurring-candidate selection. |
| `merchant_amount_deviation` | Section 9: `robust_zscores` per merchant (not just per category) could catch a merchant charging an unusual amount for *itself*, independent of category baselines. |
| `days_since_previous_transaction` | Section 8: the raw ingredient for `median_interval_days`; useful directly as a per-transaction feature, not just a merchant-level summary. |
| `transaction_interval` | Section 8: the recurring-candidate table needs 3+ occurrences per merchant to compute `interval_variation` meaningfully — this dataset only supports the median, not the variation. |
| `monthly_category_spend` | Section 6: already computed ad hoc for the category-trend chart; worth formalising once more months exist. |
| `monthly_spend_change` | Section 4: `net_change` was computed but explicitly flagged as unreliable while March is a partial month — a proper feature needs complete-month guarding. |
| `income_change` | Section 10: only two income events exist to compare; not yet meaningful, but the right shape of feature once more exist. |
| `savings_rate` | Section 10: already computed at the monthly level; a natural candidate once enough months exist to trend it. |

## Out of Scope confirmation

No production anomaly detector, ML model, clustering, forecasting,
automated categorisation, LLM-generated insight or recommendation was
implemented in this notebook — only exploratory functions and an
exploratory candidate table for recurring transactions, per PR-008.